# 04. 앙상블 학습 — 회귀 실습 (RandomForest · XGBoost · Stacking)
> 교재 92~113p · SK네트웍스 Family AI 캠프 37기 · 2026-10-14

**회귀 문제의 앙상블**: 개별 예측값이 연속값이라 투표(다수결)가 불가능 → 개별 회귀 모델 예측값의 **평균**으로 최종 예측

| 방식 | 회귀 모델 |
|---|---|
| Bagging | `RandomForestRegressor` |
| Boosting | `XGBRegressor` |
| Stacking | `StackingRegressor` (메타 모델: `LinearRegression`) |

In [1]:
# 필요한 라이브러리 임포트
import pandas as pd

# 경고 메시지 숨기기
import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 불러오기 — 보스턴 집값 (정답: `medv`)

In [2]:
# 노트북과 같은 폴더에 house_price.csv를 두고 실행 (다른 위치라면 경로만 수정)
file_path = './house_price.csv'

df = pd.read_csv(file_path)
print(f'데이터 크기: {df.shape}')   # (506, 14)
display(df.head())

데이터 크기: (506, 14)


,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


## 2. 학습용 / 평가용 데이터 생성 (75 : 25)

In [3]:
from sklearn.model_selection import train_test_split

X_data = df.drop(columns=['medv'])
y_data = df['medv']

# random_state 고정 -> 모델끼리 같은 데이터로 공정하게 비교
# 회귀(연속값)는 stratify를 쓰지 않는다
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data,
    test_size=0.25,
    random_state=0
)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

X_train: (379, 13), X_test: (127, 13)


> 💡 트리 기반 모델(RandomForest, XGBoost)은 특성 값의 **크기 비교**로 분할하므로
> 표준화(StandardScaler)를 하지 않아도 성능에 영향이 거의 없다.

## 3. RandomForest 회귀 모델 (Bagging)

In [4]:
from sklearn.ensemble import RandomForestRegressor
# 회귀 평가는 accuracy_score가 아니라 RMSE (오차가 작을수록 좋음)
from sklearn.metrics import root_mean_squared_error as rmse

# 모델 생성 · 학습
rf = RandomForestRegressor(random_state=0)
rf.fit(X_train, y_train)

# 예측 · 평가
pred_test_rf = rf.predict(X_test)
rmse_rf = rmse(y_test, pred_test_rf)
print(f'RandomForest RMSE: {rmse_rf:.4f}')

RandomForest RMSE: 4.0720


## 4. XGBoost 회귀 모델 (Boosting)
⚠️ `XGBRFRegressor`(XGBoost로 만든 **랜덤 포레스트**)가 아니라 **`XGBRegressor`**(부스팅)를 사용해야 한다.

In [5]:
from xgboost import XGBRegressor

# 모델 생성 · 학습
xgb = XGBRegressor(random_state=0)
xgb.fit(X_train, y_train)

# 예측 · 평가
pred_test_xgb = xgb.predict(X_test)
rmse_xgb = rmse(y_test, pred_test_xgb)
print(f'XGBoost RMSE: {rmse_xgb:.4f}')

XGBoost RMSE: 4.2298


## 5. Stacking 회귀 모델
- 1단계: 서로 다른 기본 모델(DecisionTree, RandomForest, XGBoost) 학습 · 예측
- 2단계: 기본 모델들의 예측값을 입력(X)으로 삼아 메타 모델(LinearRegression)이 최종 예측

In [6]:
from sklearn.ensemble import StackingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

# 1단계: 개별 기본 모델(Base Estimators)
base_estimators = [
    ('dt', DecisionTreeRegressor(random_state=0)),
    ('rf', RandomForestRegressor(random_state=0)),
    ('xgb', XGBRegressor(random_state=0)),
]

# 2단계: 최종 메타 모델
final_meta_learner = LinearRegression()

# Stacking 모델 생성
stack = StackingRegressor(
    estimators=base_estimators,
    final_estimator=final_meta_learner,
    n_jobs=-1
)

In [7]:
# 모델 학습
stack.fit(X_train, y_train)

# 예측 · 평가
pred_test_stack = stack.predict(X_test)
rmse_stack = rmse(y_test, pred_test_stack)
print(f'Stacking RMSE: {rmse_stack:.4f}')

Stacking RMSE: 4.1951


In [8]:
# 메타 모델(LinearRegression)이 각 기본 모델에 준 가중치
# -> 어떤 모델의 예측을 더 신뢰했는지 확인할 수 있다
weights = pd.Series(stack.final_estimator_.coef_, index=[name for name, _ in base_estimators])
display(weights.round(3).to_frame('메타 모델 가중치'))

,메타 모델 가중치
dt,0.079
rf,0.098
xgb,0.854


## 6. 결과 비교

In [9]:
result = pd.DataFrame({
    'model': ['RandomForest', 'XGBoost', 'Stacking'],
    'RMSE': [rmse_rf, rmse_xgb, rmse_stack],
}).set_index('model').sort_values('RMSE')   # RMSE는 작을수록 좋음 -> 오름차순

display(result.round(4))

,RMSE
model,
RandomForest,4.0720
Stacking,4.1951
XGBoost,4.2298


## ✅ 정리
- 회귀 앙상블은 투표 대신 **평균**, 평가지표는 **RMSE**(작을수록 좋음)
- 실행 결과(random_state=0): RandomForest ≈ 4.07 · Stacking ≈ 4.20 · XGBoost ≈ 4.23
- 선형 회귀(RMSE ≈ 5.46)보다 트리 앙상블의 오차가 작다 → 비선형 관계를 잘 잡아냄
- 메타 모델 가중치를 보면 XGBoost 예측을 가장 많이 반영했지만, 최종 성능은 단일 RandomForest가 가장 좋았다
- 부스팅 회귀는 `XGBRegressor`, `XGBRFRegressor`는 랜덤 포레스트 방식이니 구분할 것
- `train_test_split`에 `random_state`를 고정해야 모델 간 비교가 공정하다